# Notebook 03 — Stratified Train / Val / Test Split (6 Classes)

Split at SA level. All 6 classes in all 3 splits.

**Ratio:** 70% train | 15% val | 15% test


## 0. Imports and paths

In [1]:
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

BASE_DIR  = Path(r"E:\THESIS_COLLINS HLORDZIE\02_PROCESSED")
TILES_DIR = BASE_DIR / "Tiles"
LABEL_DIR = BASE_DIR / "Labels"

RGB_IMG_DIR = TILES_DIR / "rgb" / "images"
LBL_DIR     = TILES_DIR / "labels"

SKIP_SAS   = {"SA012_10445982"}
BACKGROUND = 255
N_CLASSES  = 6

CLASS_NAMES = [
    "Maize",
    "Maize+Pumpkin",
    "Beans+Maize",
    "Cassava+Maize",
    "Grass",
    "Mixed"
]

print("Config ready.")
print("Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i} = {name}")

Config ready.
Classes:
  0 = Maize
  1 = Maize+Pumpkin
  2 = Beans+Maize
  3 = Cassava+Maize
  4 = Grass
  5 = Mixed


## 1. Load class presence per SA from label rasters

In [2]:
rows = []
label_files = sorted(LABEL_DIR.glob("*_labels.tif"))
print(f"Found {len(label_files)} label rasters.", flush=True)

for label_path in label_files:
    sa_id = label_path.stem.replace("_labels", "")
    with rasterio.open(label_path) as src:
        data = src.read(1).ravel()
    row = {"SA": sa_id}
    for cid, cname in enumerate(CLASS_NAMES):
        row[cname] = int(np.sum(data == cid))
    row["Total_valid"] = int(np.sum(data != BACKGROUND))
    rows.append(row)

presence_df = pd.DataFrame(rows).set_index("SA")

print("\nClass presence per SA (pixel counts):")
display(presence_df)

print("\nSAs containing each class (> 0 pixels):")
for cname in CLASS_NAMES:
    sas = presence_df[presence_df[cname] > 0].index.tolist()
    print(f"  {cname:20s} -> {len(sas)} SAs: {sas}")

Found 19 label rasters.

Class presence per SA (pixel counts):


,Maize,Maize+Pumpkin,Beans+Maize,Cassava+Maize,Grass,Mixed,Total_valid
SA,,,,,,,
10085703,18573922,942586,0,0,0,0,19516508
10125706,2070139,2459444,0,0,47993004,870357,53392944
10145814,20430756,5303525,0,0,27550879,5470996,58756156
10165835,12084074,0,0,0,0,0,12084074
10165859,9961576,0,0,0,0,1423707,11385283
10185850,12779196,2330958,0,0,0,1046345,16156499
10305790,58736656,0,334996,254914,0,497776,59824342
10445787,10027122,1682978,0,0,0,6057220,17767320
10565691,54649,6814729,0,0,62600855,0,69470233



SAs containing each class (> 0 pixels):
  Maize                -> 17 SAs: ['10085703', '10125706', '10145814', '10165835', '10165859', '10185850', '10305790', '10445787', '10565691', '10595716', '10765685', '10865406', '10865805', '11125706', '9545448', '9805753', '9965805']
  Maize+Pumpkin        -> 11 SAs: ['10085703', '10125706', '10145814', '10185850', '10445787', '10565691', '10595716', '10765685', '10865805', '9545448', '9805753']
  Beans+Maize          -> 6 SAs: ['10305790', '10765685', '10865406', '10865805', '11125706', '9545448']
  Cassava+Maize        -> 5 SAs: ['10305790', '10595716', '10865805', '11125706', '9545448']
  Grass                -> 5 SAs: ['10125706', '10145814', '10565691', '10595716', '10605304']
  Mixed                -> 15 SAs: ['10125706', '10145814', '10165859', '10185850', '10305790', '10445787', '10605304', '10765685', '10865406', '10865805', '11125706', '11226291', '9545448', '9805753', '9965805']


## 2. Constrained split assignment

Every class must appear in train, val AND test.

Constraints:
- Grass (5 SAs): 10125706, 10145814, 10565691, 10595716, 10605304
- Cassava+Maize (5 SAs): 10305790, 10595716, 10865805, 11125706, 9545448
- Beans+Maize (6 SAs): 10305790, 10765685, 10865406, 10865805, 11125706, 9545448

In [3]:
FORCED = {
    "11226291": "train",
    "9965805":  "train",
    "9805753":  "train",
    "10565691": "train",
    "10305790": "train",
    "10865406": "train",
    "10165835": "train",
    "10085703": "train",
    "10185850": "train",
    "9545448":  "train",
    "10445787": "train",
    "11125706": "train",
    "10165859": "train",
    "10145814": "val",
    "10595716": "val",
    "10765685": "val",
    "10125706": "test",
    "10605304": "test",
    "10865805": "test"
}

all_sa_ids = sorted([
    d.name.split("_", 1)[-1]
    for d in BASE_DIR.iterdir()
    if d.is_dir() and d.name.startswith("SA") and d.name not in SKIP_SAS
])

print(f"Total SAs found: {len(all_sa_ids)}", flush=True)
missing = [sa for sa in all_sa_ids if sa not in FORCED]
if missing:
    print(f"WARNING - unassigned SAs: {missing}")
else:
    print("All SAs assigned.\n")

assignment = FORCED.copy()

for split in ["train", "val", "test"]:
    assigned = sorted([sa for sa, s in assignment.items() if s == split])
    print(f"{split.upper():6s} ({len(assigned)} SAs): {', '.join(assigned)}")

Total SAs found: 19
All SAs assigned.

TRAIN  (13 SAs): 10085703, 10165835, 10165859, 10185850, 10305790, 10445787, 10565691, 10865406, 11125706, 11226291, 9545448, 9805753, 9965805
VAL    (3 SAs): 10145814, 10595716, 10765685
TEST   (3 SAs): 10125706, 10605304, 10865805


## 3. Verify class coverage across all splits

Must show OK for every class in every split before proceeding.

In [4]:
print("=" * 60)
print("Class coverage verification")
print("=" * 60)

all_ok = True
rows = []

for cname in CLASS_NAMES:
    sas_with_class = presence_df[presence_df[cname] > 0].index.tolist()
    row = {"Class": cname}
    for split in ["train", "val", "test"]:
        split_sas = [sa for sa, s in assignment.items() if s == split]
        has_class = any(sa in sas_with_class for sa in split_sas)
        row[split] = "OK" if has_class else "MISSING"
        if not has_class:
            all_ok = False
    rows.append(row)

coverage_df = pd.DataFrame(rows).set_index("Class")
display(coverage_df)

if all_ok:
    print("\nAll classes present in all three splits. Safe to proceed.")
else:
    print("\nWARNING - fix assignments before moving tiles.")

Class coverage verification


,train,val,test
Class,,,
Maize,OK,OK,OK
Maize+Pumpkin,OK,OK,OK
Beans+Maize,OK,OK,OK
Cassava+Maize,OK,OK,OK
Grass,OK,OK,OK
Mixed,OK,OK,OK



All classes present in all three splits. Safe to proceed.


## 4. Create split sub-directories



In [5]:
splits = ["train", "val", "test"]

for split in splits:
    (RGB_IMG_DIR / split).mkdir(parents=True, exist_ok=True)
    (LBL_DIR     / split).mkdir(parents=True, exist_ok=True)
    print(f"  Created: {RGB_IMG_DIR / split}")
    print(f"  Created: {LBL_DIR     / split}")

print("\nSplit directories ready.")

  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\rgb\images\train
  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\labels\train
  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\rgb\images\val
  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\labels\val
  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\rgb\images\test
  Created: E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\labels\test

Split directories ready.


## 5. Move tiles into split folders

Note: Tiles are MOVED not copied. Run once only.
If you need to re-run, go back to Notebook 2 and re-tile first.

In [6]:
def get_sa_id(tile_name):
    return tile_name.split("_r")[0]


def move_tiles(src_dir, split_map, stream_name):
    tiles   = sorted(src_dir.glob("*.tif"))
    moved   = 0
    skipped = 0
    for tile in tiles:
        sa_id = get_sa_id(tile.name)
        split = assignment.get(sa_id)
        if split is None:
            print(f"  [WARN] No assignment for {sa_id}", flush=True)
            skipped += 1
            continue
        dest = split_map[split] / tile.name
        shutil.move(str(tile), str(dest))
        moved += 1
    print(f"  {stream_name}: {moved:,} moved, {skipped} skipped", flush=True)


print("Moving RGB tiles ...", flush=True)
move_tiles(
    RGB_IMG_DIR,
    {s: RGB_IMG_DIR / s for s in splits},
    "RGB"
)

print("Moving label tiles ...", flush=True)
move_tiles(
    LBL_DIR,
    {s: LBL_DIR / s for s in splits},
    "Labels"
)

print("\nDone.")

Moving RGB tiles ...
  RGB: 22,191 moved, 0 skipped
Moving label tiles ...
  Labels: 22,191 moved, 0 skipped

Done.


## 6. Final summary and verification

In [7]:
rows = []
for split in splits:
    img_tiles = list((RGB_IMG_DIR / split).glob("*.tif"))
    lbl_tiles = list((LBL_DIR     / split).glob("*.tif"))
    sa_list   = sorted({get_sa_id(t.name) for t in img_tiles})
    match     = "OK" if len(img_tiles) == len(lbl_tiles) else "MISMATCH"
    rows.append({
        "split":   split,
        "n_SAs":   len(sa_list),
        "n_tiles": len(img_tiles),
        "labels":  len(lbl_tiles),
        "match":   match,
        "SAs":     ", ".join(sa_list),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(TILES_DIR / "split_summary.csv", index=False)

print("Split summary:")
display(summary_df)

total = summary_df["n_tiles"].sum()
print(f"\nTotal tiles: {total:,}")
print(f"Summary saved -> {TILES_DIR / 'split_summary.csv'}")

Split summary:


,split,n_SAs,n_tiles,labels,match,SAs
0,train,13,11915,11915,OK,"10085703, 10165835, 10165859, 10185850, 103057..."
1,val,3,5200,5200,OK,"10145814, 10595716, 10765685"
2,test,3,5076,5076,OK,"10125706, 10605304, 10865805"



Total tiles: 22,191
Summary saved -> E:\THESIS_COLLINS HLORDZIE\02_PROCESSED\Tiles\split_summary.csv
